In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load the text files from your folder
# Replace 'my_txt_files' with the actual path to your folder containing the .txt files
folder_path = "my_txt_files" 
documents = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        # Using utf-8 encoding is crucial for reading Arabic characters properly
        loader = TextLoader(file_path, encoding="utf-8")
        documents.extend(loader.load())

# 2. Split documents into manageable chunks
# Since AraBERT has a 512-token limit, keeping chunk sizes around 300-400 characters works well
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# 3. Initialize AraBERT Embeddings
# Using aubmindlab's AraBERT v0.2 model wrapped in HuggingFaceEmbeddings
model_name = "aubmindlab/bert-base-arabertv02"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'} # Change to 'cuda' if you have a GPU available
)

# 4. Initialize Chroma Database (matching your exact configuration)
db = Chroma(
    embedding_function=embeddings,
    collection_name="my_rag_project",
    persist_directory="./my_vector_db",
    collection_metadata={"hnsw:space": "cosine"} # CRUCIAL FOR ARABIC SEMANTICS
)

# 5. Add documents in small, bite-sized batches of 10 chunks
batch_size = 10
for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    db.add_documents(batch)
    print(f"Successfully processed chunks {i} to {i + len(batch)}")

print("==== Ingestion Complete! Data saved permanently to disk ====")


C:\Users\User\AppData\Local\Temp\ipykernel_28864\773898149.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


ModuleNotFoundError: No module named 'langchain_huggingface'